<a href="https://colab.research.google.com/github/anushagtalwar74-pixel/flyrank-ml-internW1/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nipun-Wanjale-dev/ML-Internship-NSW/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN:
    print("✅ Token loaded successfully")
else:
    print("❌ Token not found")

✅ Token loaded successfully


In [ ]:
!pip -q install datasets duckdb huggingface_hub pyarrow pandas

In [ ]:
from datasets import load_dataset
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=HF_TOKEN
)

print(next(iter(ds)))

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

{'report_date': datetime.date(2025, 1, 27), 'client_hash_id': 'client_9958f0a7ae1df715', 'content_hash_id': 'content_3b70a18ea133b2bb', 'client_has_gsc': True, 'client_has_ga4': True, 'gsc_data_available': True, 'ga4_data_available': False, 'gsc_impressions': 30, 'gsc_clicks': 0, 'gsc_sum_position': 115, 'gsc_avg_position': 3.8333333333333335, 'ga4_pageviews': 0, 'ga4_sessions': 0, 'ga4_users': 0, 'ga4_engaged_sessions': 0, 'ga4_total_engagement_sec': 0, 'sessions_organic': 0, 'sessions_direct': 0, 'sessions_referral': 0, 'sessions_social': 0, 'sessions_paid': 0, 'sessions_ai': 0, 'ai_chatgpt': 0, 'ai_perplexity': 0, 'ai_gemini': 0, 'ai_copilot': 0, 'ai_claude': 0, 'ai_meta': 0, 'ai_other': 0, 'scroll_events': 0}


# My Rule

For my Content Refresh Prioritization lane, I will rank pages based on their Google Search performance. Pages with high impressions but low click-through rate (CTR) will receive a higher score because they are visible in search but may benefit from refreshed content or improved titles.

## Reason Codes

**LOW_CTR**
- The page has many impressions but few clicks.

**HIGH_IMPRESSIONS**
- The page receives strong search visibility.

**REFRESH_CANDIDATE**
- The page is recommended for content refresh based on the baseline score.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import pandas as pd

rows = []

for row in ds:

    impressions = row["gsc_impressions"]
    clicks = row["gsc_clicks"]

    ctr = clicks / impressions if impressions > 0 else 0

    score = impressions * (1 - ctr)

    if ctr < 0.05:
        reason = "LOW_CTR"
    else:
        reason = "HIGH_IMPRESSIONS"

    rows.append({
        "content_hash_id": row["content_hash_id"],
        "score": score,
        "reason_code": reason,
        "action": "Refresh Content"
    })

    # Limit rows so the notebook runs quickly
    if len(rows) >= 500:
        break

df = pd.DataFrame(rows)

df = df.sort_values(by="score", ascending=False)

df.head(10)

,content_hash_id,score,reason_code,action
446,content_f94fe855380e150f,300.0,LOW_CTR,Refresh Content
146,content_f94fe855380e150f,260.0,LOW_CTR,Refresh Content
349,content_d02be57d816cf3d7,173.0,LOW_CTR,Refresh Content
45,content_d02be57d816cf3d7,118.0,LOW_CTR,Refresh Content
76,content_de7b08874af74c00,79.0,LOW_CTR,Refresh Content
383,content_84f5a9ecfefa108e,78.0,LOW_CTR,Refresh Content
91,content_5e770041ee8f2231,70.0,LOW_CTR,Refresh Content
89,content_f3a75d8cf58dd50b,66.0,LOW_CTR,Refresh Content
80,content_84f5a9ecfefa108e,65.0,LOW_CTR,Refresh Content
348,content_c4dca383bdc83897,62.0,LOW_CTR,Refresh Content


In [ ]:
df.head(10)

,content_hash_id,score,reason_code,action
446,content_f94fe855380e150f,300.0,LOW_CTR,Refresh Content
146,content_f94fe855380e150f,260.0,LOW_CTR,Refresh Content
349,content_d02be57d816cf3d7,173.0,LOW_CTR,Refresh Content
45,content_d02be57d816cf3d7,118.0,LOW_CTR,Refresh Content
76,content_de7b08874af74c00,79.0,LOW_CTR,Refresh Content
383,content_84f5a9ecfefa108e,78.0,LOW_CTR,Refresh Content
91,content_5e770041ee8f2231,70.0,LOW_CTR,Refresh Content
89,content_f3a75d8cf58dd50b,66.0,LOW_CTR,Refresh Content
80,content_84f5a9ecfefa108e,65.0,LOW_CTR,Refresh Content
348,content_c4dca383bdc83897,62.0,LOW_CTR,Refresh Content


In [ ]:
import os

os.makedirs("work/outputs", exist_ok=True)

df.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("✅ CSV saved successfully!")

df.head(10)

✅ CSV saved successfully!


,content_hash_id,score,reason_code,action
446,content_f94fe855380e150f,300.0,LOW_CTR,Refresh Content
146,content_f94fe855380e150f,260.0,LOW_CTR,Refresh Content
349,content_d02be57d816cf3d7,173.0,LOW_CTR,Refresh Content
45,content_d02be57d816cf3d7,118.0,LOW_CTR,Refresh Content
76,content_de7b08874af74c00,79.0,LOW_CTR,Refresh Content
383,content_84f5a9ecfefa108e,78.0,LOW_CTR,Refresh Content
91,content_5e770041ee8f2231,70.0,LOW_CTR,Refresh Content
89,content_f3a75d8cf58dd50b,66.0,LOW_CTR,Refresh Content
80,content_84f5a9ecfefa108e,65.0,LOW_CTR,Refresh Content
348,content_c4dca383bdc83897,62.0,LOW_CTR,Refresh Content


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The ranked pages were selected because they have high search impressions but a very low click-through rate (CTR). These pages may benefit from content updates, better titles, improved meta descriptions, or refreshed information.

| Action | Reason Code | Confidence | What would make it wrong? |
|--------|-------------|------------|---------------------------|
| Refresh Content | LOW_CTR | Medium | Low CTR may be caused by poor ranking instead of outdated content. |
| Refresh Content | LOW_CTR | Medium | Seasonal search trends may affect clicks. |
| Refresh Content | LOW_CTR | Medium | Search intent may have changed. |
| Refresh Content | LOW_CTR | Medium | The page may already be optimized. |
| Refresh Content | LOW_CTR | Medium | Competitor pages may perform better. |
| Refresh Content | LOW_CTR | Medium | Low CTR alone may not require refreshing. |
| Refresh Content | LOW_CTR | Medium | Limited historical data may affect ranking. |
| Refresh Content | LOW_CTR | Medium | External events may influence traffic. |
| Refresh Content | LOW_CTR | Medium | User behavior may change over time. |
| Refresh Content | LOW_CTR | Medium | More features are needed for better confidence. |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may appear in the ranked list because they have high impressions but naturally low CTR. These pages are not always true refresh candidates.

## Leakage Check

No future information or label-derived columns were used while calculating the baseline score.

The baseline only uses:
- Google Search impressions
- Google Search clicks
- Calculated CTR

Therefore, no data leakage is present in this baseline model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.